# Phase 2: Data Understanding and Engineering

This notebook documents the second phase of the Bosch Production Line Performance project.

Phase 2 objectives:

7. Parse feature names into Line, Station, and Feature IDs.
8. Create metadata tables for production lines and stations.
9. Analyze missing value patterns.
10. Create station presence indicators.
11. Generate feature completeness metrics.
12. Build a manufacturing flow dataset.

The notebook uses the reusable script `src/data/phase2_data_understanding_engineering.py`, then previews the resulting tables.

## 1. Import Libraries and Project Code

We add the project root to `sys.path` so the notebook can import the reusable Phase 2 functions from `src/data`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import pyarrow.parquet as pq

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.data.phase2_data_understanding_engineering import (
    REPORTS_DIR,
    PROCESSED_DIR,
    build_completeness_metrics,
    build_feature_metadata,
    build_manufacturing_flow_dataset,
    build_metadata_tables,
    write_report,
)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

print('Project root:', PROJECT_ROOT)
print('Reports folder:', REPORTS_DIR)
print('Processed folder:', PROCESSED_DIR)

Project root: C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch Production Line Performance
Reports folder: C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch Production Line Performance\reports
Processed folder: C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch Production Line Performance\data\processed


## 2. Parse Bosch Feature Names

Bosch feature names follow a structured pattern:

```text
L{line}_S{station}_F{feature_id}
L{line}_S{station}_D{feature_id}
```

`L` identifies the production line, `S` identifies the station, and `F` or `D` identifies feature/date measurements. Parsing this pattern gives us a manufacturing-aware metadata layer.

In [2]:
feature_metadata = build_feature_metadata()

print('Parsed rows:', len(feature_metadata))
print('Unique feature columns:', feature_metadata['column'].nunique())
feature_metadata.head(10)

Parsed rows: 8528
Unique feature columns: 4264


,dataset,split,data_type,column,line,station,station_key,feature_kind,feature_id
0,train_numeric,train,numeric,L0_S0_F0,0,0,L0_S0,F,0
1,train_numeric,train,numeric,L0_S0_F2,0,0,L0_S0,F,2
2,train_numeric,train,numeric,L0_S0_F4,0,0,L0_S0,F,4
3,train_numeric,train,numeric,L0_S0_F6,0,0,L0_S0,F,6
4,train_numeric,train,numeric,L0_S0_F8,0,0,L0_S0,F,8
5,train_numeric,train,numeric,L0_S0_F10,0,0,L0_S0,F,10
6,train_numeric,train,numeric,L0_S0_F12,0,0,L0_S0,F,12
7,train_numeric,train,numeric,L0_S0_F14,0,0,L0_S0,F,14
8,train_numeric,train,numeric,L0_S0_F16,0,0,L0_S0,F,16
9,train_numeric,train,numeric,L0_S0_F18,0,0,L0_S0,F,18


## 3. Create Production Line and Station Metadata

The station metadata table summarizes which lines and stations exist and how many numeric, categorical, and date features are attached to each station.

In [3]:
station_metadata, line_metadata = build_metadata_tables(feature_metadata)

display(line_metadata)
station_metadata.head(15)

,line,station_count,numeric_feature_count,categorical_feature_count,date_feature_count,total_feature_count
0,0,24,168,323,184,675
1,1,2,513,1227,621,2361
2,2,3,42,159,78,279
3,3,23,245,431,273,949


,line,station,station_key,categorical_feature_count,date_feature_count,numeric_feature_count,total_feature_count,has_numeric,has_categorical,has_date
0,0,0,L0_S0,0,12,12,24,True,False,True
1,0,1,L0_S1,4,2,2,8,True,True,True
2,0,2,L0_S2,18,9,9,36,True,True,True
3,0,3,L0_S3,18,9,9,36,True,True,True
4,0,4,L0_S4,6,2,2,10,True,True,True
5,0,5,L0_S5,0,2,2,4,True,False,True
6,0,6,L0_S6,10,5,3,18,True,True,True
7,0,7,L0_S7,0,5,3,8,True,False,True
8,0,8,L0_S8,0,4,3,7,True,False,True
9,0,9,L0_S9,39,13,12,64,True,True,True


## 4. Analyze Missing-Value Patterns and Completeness

Phase 1 already scanned every large CSV and saved per-column missing values. Phase 2 reuses that file, attaches line/station metadata, and creates completeness metrics at feature and station levels.

In [4]:
feature_completeness, station_completeness = build_completeness_metrics(feature_metadata)

print('Feature completeness rows:', len(feature_completeness))
print('Station completeness rows:', len(station_completeness))

station_completeness.sort_values('completeness_pct').head(15)

Feature completeness rows: 8528
Station completeness rows: 272


,split,data_type,line,station,station_key,feature_count,rows,missing_values,observed_values,possible_values,completeness_pct,missing_pct
25,test,categorical,3,36,L3_S36,8,1183748,9469984,0,9469984,0.0000,100.0000
27,test,categorical,3,39,L3_S39,8,1183748,9469984,0,9469984,0.0000,100.0000
31,test,categorical,3,46,L3_S46,3,1183748,3551244,0,3551244,0.0000,100.0000
167,train,categorical,3,46,L3_S46,3,1183747,3551240,1,3551241,0.0000,100.0000
150,train,categorical,0,23,L0_S23,30,1183747,35512410,0,35512410,0.0000,100.0000
147,train,categorical,0,18,L0_S18,10,1183747,11837470,0,11837470,0.0000,100.0000
138,train,categorical,0,3,L0_S3,18,1183747,21307446,0,21307446,0.0000,100.0000
80,test,date,3,46,L3_S46,1,1183748,1183748,0,1183748,0.0000,100.0000
9,test,categorical,0,15,L0_S15,9,1183748,10653732,0,10653732,0.0000,100.0000
145,train,categorical,0,15,L0_S15,9,1183747,10653714,9,10653723,0.0001,99.9999


The most complete station/data-type groups are also useful because they show where we have the strongest signal density for later modeling.

In [5]:
station_completeness.sort_values('completeness_pct', ascending=False).head(15)

,split,data_type,line,station,station_key,feature_count,rows,missing_values,observed_values,possible_values,completeness_pct,missing_pct
259,train,numeric,3,37,L3_S37,4,1183747,253412,4481576,4734988,94.6481,5.3519
207,train,date,3,37,L3_S37,6,1183747,380118,6722364,7102482,94.6481,5.3519
123,test,numeric,3,37,L3_S37,4,1183748,253928,4481064,4734992,94.6372,5.3628
71,test,date,3,37,L3_S37,6,1183748,380892,6721596,7102488,94.6372,5.3628
199,train,date,3,29,L3_S29,63,1183747,4190349,70385712,74576061,94.3811,5.6189
63,test,date,3,29,L3_S29,63,1183748,4203018,70373106,74576124,94.3641,5.6359
251,train,numeric,3,29,L3_S29,53,1183747,3549169,59189422,62738591,94.3429,5.6571
115,test,numeric,3,29,L3_S29,53,1183748,3559908,59178736,62738644,94.3258,5.6742
256,train,numeric,3,34,L3_S34,4,1183747,274516,4460472,4734988,94.2024,5.7976
204,train,date,3,34,L3_S34,5,1183747,343145,5575590,5918735,94.2024,5.7976


## 5. Save Metadata and Completeness Tables

These CSV outputs are compact and reusable in later EDA/modeling notebooks.

In [6]:
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

feature_metadata.to_csv(REPORTS_DIR / 'phase2_feature_metadata.csv', index=False)
station_metadata.to_csv(REPORTS_DIR / 'phase2_station_metadata.csv', index=False)
line_metadata.to_csv(REPORTS_DIR / 'phase2_line_metadata.csv', index=False)
feature_completeness.to_csv(REPORTS_DIR / 'phase2_feature_completeness_metrics.csv', index=False)
station_completeness.to_csv(REPORTS_DIR / 'phase2_station_completeness_metrics.csv', index=False)

print('Saved Phase 2 metadata and completeness tables.')

Saved Phase 2 metadata and completeness tables.


## 6. Create Station Presence Indicators

Station presence is derived from date features. In this dataset, a non-null date feature is a good signal that a part appeared at that station.

For every `Id`, the engineered flow dataset includes:

- One `present_Lx_Sy` indicator per station.
- One line-level presence indicator per line.
- Count of stations visited.
- Count of production lines touched.
- First and last station IDs observed.
- Date-feature completeness for that part.
- `Response` for train rows only.

In [7]:
flow_paths = {
    'train': build_manufacturing_flow_dataset('train', feature_metadata, chunksize=20_000),
    'test': build_manufacturing_flow_dataset('test', feature_metadata, chunksize=20_000),
}

flow_paths

Building train flow: 60chunk [02:36,  2.61s/chunk]
Building test flow: 60chunk [01:53,  1.89s/chunk]


{'train': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/data/processed/manufacturing_flow_train.parquet'),
 'test': WindowsPath('C:/Users/karin/OneDrive/Desktop/Nexturn/Bosch Production Line Performance/data/processed/manufacturing_flow_test.parquet')}

## 7. Preview Manufacturing Flow Dataset

The final flow dataset is stored as Parquet because it is much more efficient than CSV for wide engineered tables.

In [8]:
train_flow_preview = pd.read_parquet(flow_paths['train']).head(10)
test_flow_preview = pd.read_parquet(flow_paths['test']).head(10)

print('Train flow columns:', len(train_flow_preview.columns))
display(train_flow_preview)

print('Test flow columns:', len(test_flow_preview.columns))
display(test_flow_preview)

Train flow columns: 65


,Id,Response,present_L0_S0,present_L0_S1,present_L0_S2,present_L0_S3,present_L0_S4,present_L0_S5,present_L0_S6,present_L0_S7,present_L0_S8,present_L0_S9,present_L0_S10,present_L0_S11,present_L0_S12,...,present_L3_S48,present_L3_S49,present_L3_S50,present_L3_S51,line_0_present,line_1_present,line_2_present,line_3_present,station_count,line_count,first_station,last_station,observed_date_values,possible_date_values,date_completeness_pct
0,4,0,1,1,1,0,1,0,0,1,1,0,0,1,0,...,0,0,0,0,1,0,0,1,14,2,0,37,179,1156,15.484429
1,6,0,0,0,0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,13,2,12,37,209,1156,18.079584
2,7,0,1,1,1,0,0,1,1,0,1,0,1,0,0,...,0,0,0,0,1,0,0,1,13,2,0,37,207,1156,17.906574
3,9,0,1,1,1,0,1,0,0,1,1,0,1,0,0,...,0,0,0,0,1,0,0,1,13,2,0,37,207,1156,17.906574
4,11,0,1,1,0,1,1,0,0,1,1,0,0,1,0,...,0,0,0,0,1,0,0,1,13,2,0,37,207,1156,17.906574
5,13,0,1,1,0,1,1,0,0,1,1,0,1,0,0,...,0,0,0,0,1,0,0,1,13,2,0,37,207,1156,17.906574
6,14,0,0,0,0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,13,2,12,37,209,1156,18.079584
7,16,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,8,3,24,37,249,1156,21.539793
8,18,0,1,1,1,0,1,0,0,1,1,0,1,0,0,...,0,0,0,0,1,0,0,1,13,2,0,37,175,1156,15.138409
9,23,0,0,0,0,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,1,0,0,1,13,2,12,37,177,1156,15.311419


Test flow columns: 64


,Id,present_L0_S0,present_L0_S1,present_L0_S2,present_L0_S3,present_L0_S4,present_L0_S5,present_L0_S6,present_L0_S7,present_L0_S8,present_L0_S9,present_L0_S10,present_L0_S11,present_L0_S12,present_L0_S13,...,present_L3_S48,present_L3_S49,present_L3_S50,present_L3_S51,line_0_present,line_1_present,line_2_present,line_3_present,station_count,line_count,first_station,last_station,observed_date_values,possible_date_values,date_completeness_pct
0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,8,3,24,37,246,1156,21.280277
1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,8,3,24,37,262,1156,22.664360
2,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,8,3,24,37,249,1156,21.539793
3,5,1,1,1,0,1,0,0,1,1,0,1,0,0,0,...,0,0,0,0,1,0,1,1,14,3,0,37,201,1156,17.387543
4,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,8,3,24,37,249,1156,21.539793
5,10,1,1,1,0,0,1,0,1,1,1,0,0,0,0,...,0,0,0,0,1,0,0,1,13,2,0,37,207,1156,17.906574
6,12,1,1,0,1,1,0,0,1,1,0,0,1,0,0,...,0,0,0,0,1,0,0,1,14,2,0,38,210,1156,18.166090
7,15,1,1,0,1,0,1,1,0,1,0,1,0,0,0,...,0,0,0,0,1,0,0,1,13,2,0,37,207,1156,17.906574
8,17,0,0,0,0,0,0,0,0,0,0,0,0,1,1,...,1,1,0,1,1,0,0,1,16,2,12,51,122,1156,10.553634
9,19,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,1,1,8,3,24,37,248,1156,21.453287


## 8. Save Phase 2 Report

This creates a markdown report summarizing the metadata, completeness patterns, and engineered manufacturing flow outputs.

In [11]:
write_report(
    feature_metadata=feature_metadata,
    station_metadata=station_metadata,
    line_metadata=line_metadata,
    station_completeness=station_completeness,
    flow_paths=flow_paths,
)

print('Saved:', REPORTS_DIR / 'phase2_data_understanding_engineering_report.md')

Saved: C:\Users\karin\OneDrive\Desktop\Nexturn\Bosch Production Line Performance\reports\phase2_data_understanding_engineering_report.md


## 9. Phase 2 Deliverables

After running this notebook, the project has:

- Parsed feature metadata by line, station, feature type, and feature ID.
- Production line metadata.
- Station metadata.
- Feature-level completeness metrics.
- Station-level completeness and missing-value pattern metrics.
- Train and test manufacturing flow datasets for downstream modeling.

These outputs prepare the project for Phase 3, where we can explore target imbalance, station paths, line behavior, and predictive feature groups.